# APQ-Lite — Results Visualization

Use this notebook (VS Code / Jupyter) to analyse the outputs generated by the MASE terminal pipeline.

**Expected input files** (from `outputs/`):
- `sensitivity_scores.json` — raw param-level gradient norms
- `layer_sensitivity.json` — module-level aggregated scores
- `quant_config.json` — CHOP bit-width assignments
- `qat_history.json` — per-epoch training/validation metrics
- `eval_baseline.json` — fp32 model evaluation results *(optional)*
- `eval_quantized.json` — quantized model evaluation results *(optional)*

In [ ]:
import json
import pathlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
OUTPUT_DIR = pathlib.Path("../outputs")

def load_json(name: str) -> dict | list:
    path = OUTPUT_DIR / name
    if not path.exists():
        print(f"[WARNING] {path} not found — skipping.")
        return {}
    with open(path) as fh:
        return json.load(fh)

print("Imports OK")

## 1. Per-Layer Sensitivity Scores

Visualise the normalised gradient-norm sensitivity for each module.

In [ ]:
layer_scores = load_json("layer_sensitivity.json")

if layer_scores:
    # Filter to non-trivial modules (score > 0)
    df_sens = (
        pd.DataFrame(list(layer_scores.items()), columns=["layer", "sensitivity"])
        .query("sensitivity > 0")
        .sort_values("sensitivity", ascending=False)
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.barh(df_sens["layer"][::-1], df_sens["sensitivity"][::-1],
                   color=sns.color_palette("flare", len(df_sens))[::-1])
    ax.axvline(0.70, color="red",    linestyle="--", lw=1.5, label="high threshold (8-bit)")
    ax.axvline(0.35, color="orange", linestyle="--", lw=1.5, label="mid threshold (4-bit)")
    ax.set_xlabel("Normalised Sensitivity Score")
    ax.set_title("APQ-Lite — Per-Layer Gradient Sensitivity (Stage 1)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Bit-Width Distribution

Inspect the mixed-precision allocation produced by the allocator.

In [ ]:
quant_config = load_json("quant_config.json")

if quant_config:
    df_bits = pd.DataFrame(
        [
            {"layer": k, "weight_bits": v.get("weight_width", 8),
             "act_bits": v.get("data_in_width", 8)}
            for k, v in quant_config.items()
            if isinstance(v, dict)
        ]
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, col, title in zip(
        axes,
        ["weight_bits", "act_bits"],
        ["Weight Bit-Width Distribution", "Activation Bit-Width Distribution"],
    ):
        counts = df_bits[col].value_counts().sort_index()
        ax.bar(counts.index.astype(str), counts.values,
               color=sns.color_palette("Blues_r", len(counts)))
        ax.set_xlabel("Bit-width")
        ax.set_ylabel("# Layers")
        ax.set_title(title)

    plt.suptitle("APQ-Lite — Mixed-Precision Allocation (Stage 2)", fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(df_bits[["layer", "weight_bits", "act_bits"]].to_string(index=False))

## 3. QAT Training Curves

In [ ]:
history = load_json("qat_history.json")

if history:
    df_hist = pd.DataFrame(history)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss
    axes[0].plot(df_hist["epoch"], df_hist["loss"], label="train loss")
    if "val_loss" in df_hist:
        axes[0].plot(df_hist["epoch"], df_hist["val_loss"], label="val loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].set_title("QAT Loss Curve")
    axes[0].legend()

    # Accuracy
    axes[1].plot(df_hist["epoch"], df_hist["acc"] * 100, label="train acc")
    if "val_acc" in df_hist:
        axes[1].plot(df_hist["epoch"], df_hist["val_acc"] * 100, label="val acc")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("QAT Accuracy Curve")
    axes[1].legend()

    plt.suptitle("APQ-Lite — QAT Training (Stage 4)", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 4. Baseline vs. Quantized Accuracy Comparison

In [ ]:
baseline  = load_json("eval_baseline.json").get("metrics", {})
quantized = load_json("eval_quantized.json").get("metrics", {})

if baseline and quantized:
    metrics = [k for k in baseline if k.startswith("top")]
    x = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - width/2, [baseline[m]  for m in metrics], width, label="fp32 baseline")
    ax.bar(x + width/2, [quantized[m] for m in metrics], width, label="APQ-Lite (mixed)")
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Accuracy: fp32 Baseline vs. APQ-Lite Mixed-Precision")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend()
    plt.tight_layout()
    plt.show()

    for m in metrics:
        drop = baseline[m] - quantized[m]
        print(f"{m.upper()}: baseline={baseline[m]:.2f}%  quantized={quantized[m]:.2f}%  drop={drop:+.2f}%")
else:
    print("Run eval_baseline.json / eval_quantized.json first.")

## 5. Sensitivity Score vs. Assigned Bit-Width (Scatter)

Validates that the allocator policy is consistent with the profiler output.

In [ ]:
if layer_scores and quant_config:
    rows = []
    for layer, score in layer_scores.items():
        if layer in quant_config and isinstance(quant_config[layer], dict):
            rows.append({
                "layer": layer,
                "sensitivity": score,
                "weight_bits": quant_config[layer].get("weight_width", 8),
            })

    df_scatter = pd.DataFrame(rows)

    if not df_scatter.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        palette = {2: "#e74c3c", 4: "#f39c12", 8: "#2ecc71"}
        for bits, grp in df_scatter.groupby("weight_bits"):
            ax.scatter(grp["sensitivity"], [bits] * len(grp),
                       color=palette.get(bits, "grey"), label=f"{bits}-bit", s=60, alpha=0.8)
        ax.axvline(0.70, color="red",    linestyle="--", lw=1.2)
        ax.axvline(0.35, color="orange", linestyle="--", lw=1.2)
        ax.set_yticks([2, 4, 8])
        ax.set_xlabel("Normalised Sensitivity Score")
        ax.set_ylabel("Assigned Bit-Width")
        ax.set_title("Sensitivity Score vs. Assigned Bit-Width")
        ax.legend(title="bit-width")
        plt.tight_layout()
        plt.show()